# 03 - Modèles BLIP et GIT

Ce notebook est prévu pour Google Colab avec GPU. Il compare le jugement image-légende de BLIP et GIT sur les mêmes paires du notebook 02, en précision normale (fp16), 8 bits (int8) et 4 bits (int4). Les scores et mesures sont sauvegardés dans Google Drive.

Chaque expérience tourne dans sa propre cellule pour éviter que le navigateur ne se coupe pendant un long calcul. Les résultats déjà sauvegardés sont ignorés (reprise possible après une coupure).

Ce notebook ne doit pas être exécuté dans VS Code.

In [1]:
# Installer les bibliothèques nécessaires dans Google Colab.
!pip -q install datasets transformers accelerate bitsandbytes tqdm

In [2]:
# Connecter Google Drive et préparer les dossiers du projet.
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
PROJECT_DIR = Path("/content/drive/MyDrive/ia_visuelle")
RESULTS_DIR = PROJECT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

import gc
import time

import pandas as pd
import torch
from datasets import load_dataset
from transformers import BitsAndBytesConfig
from tqdm import tqdm

SEED = 42
N_IMAGES = 500
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if DEVICE != "cuda":
    raise RuntimeError("Un GPU Colab est nécessaire pour exécuter ce notebook.")

# Vérifier et réutiliser le CSV produit par le notebook 01.
references_path = RESULTS_DIR / "flickr8k_references.csv"
if not references_path.exists():
    raise FileNotFoundError(
        f"CSV introuvable : {references_path}. Exécutez le notebook 01."
    )
references = pd.read_csv(references_path)
required_columns = ["id"] + [f"caption_{number}" for number in range(5)]
missing = [column for column in required_columns if column not in references]
if missing:
    raise ValueError(f"Colonnes manquantes dans le CSV : {missing}")

# Reprendre exactement les images du notebook 01.
dataset = load_dataset("jxie/flickr8k", split="train")
sample = dataset.shuffle(seed=SEED).select(range(min(N_IMAGES, len(dataset))))
if len(references) != len(sample):
    raise ValueError("Le nombre d images ne correspond pas au notebook 01.")
for number in range(5):
    expected = [example[f"caption_{number}"] for example in sample]
    actual = references[f"caption_{number}"].tolist()
    if actual != expected:
        raise ValueError(f"La sélection caption_{number} ne correspond pas au notebook 01.")

print(f"Images utilisées : {len(sample)}")
print(f"GPU utilisé : {torch.cuda.get_device_name(0)}")

# Charger les mêmes paires correctes/incorrectes que dans le notebook 02.
pairs_path = RESULTS_DIR / "baseline_alignment_pairs.csv"
if not pairs_path.exists():
    raise FileNotFoundError(
        f"Paires introuvables : {pairs_path}. Exécutez le notebook 02."
    )
pairs_df = pd.read_csv(pairs_path)
required_pair_columns = ["image_id", "caption", "label", "pair_type"]
missing_pairs = [column for column in required_pair_columns if column not in pairs_df]
if missing_pairs:
    raise ValueError(f"Colonnes manquantes dans les paires : {missing_pairs}")
if len(pairs_df) != 2 * len(sample):
    raise ValueError("Le nombre de paires ne correspond pas au notebook 02.")
print(f"Paires image-légende évaluées : {len(pairs_df)}")

Mounted at /content/drive


README.md:   0%|          | 0.00/687 [00:00<?, ?B/s]

data/train-00000-of-00002-2f8f6bfa852eac(…): reconstructing file:   0%|          |  0.00B /  414MB            

data/train-00000-of-00002-2f8f6bfa852eac(…): downloading bytes:           |  0.00B            

data/train-00001-of-00002-2173151d8cd6c7(…): reconstructing file:   0%|          |  0.00B /  414MB            

data/train-00001-of-00002-2173151d8cd6c7(…): downloading bytes:           |  0.00B            

... (4 outputs left out)

Generating train split:   0%|          | 0/6000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Images utilisées : 500
GPU utilisé : Tesla T4
Paires image-légende évaluées : 1000


## Fonctions communes

Les fonctions suivantes chargent un modèle, calculent le score de chaque paire image-légende, mesurent le temps total et enregistrent les résultats. Une configuration déjà enregistrée est ignorée pour permettre de reprendre après une coupure.

In [3]:
def quantization_config(precision):
    """Créer la configuration bitsandbytes demandée."""
    if precision == "fp16":
        return None
    if precision == "int8":
        return BitsAndBytesConfig(load_in_8bit=True)
    if precision == "int4":
        return BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
        )
    raise ValueError(f"Précision inconnue : {precision}")


def clear_gpu():
    """Libérer le modèle précédent avant de charger le suivant."""
    gc.collect()
    torch.cuda.empty_cache()


def output_path(model_name, precision):
    return RESULTS_DIR / f"judgements_{model_name}_{precision}.csv"


def run_blip(precision):
    """Évaluer la cohérence image-légende avec BLIP."""
    from transformers import BlipForConditionalGeneration, BlipProcessor

    model_id = "Salesforce/blip-image-captioning-base"
    processor = BlipProcessor.from_pretrained(model_id)
    model = BlipForConditionalGeneration.from_pretrained(
        model_id,
        quantization_config=quantization_config(precision),
        torch_dtype=torch.float16 if precision == "fp16" else None,
        device_map="auto",
    )
    model.eval()
    torch.cuda.reset_peak_memory_stats()

    rows = []
    start = time.perf_counter()
    for _, pair in tqdm(pairs_df.iterrows(), total=len(pairs_df), desc=f"BLIP {precision}"):
        example = sample[int(pair["image_id"])]
        inputs = processor(
            images=example["image"],
            text=pair["caption"],
            return_tensors="pt",
            padding=True,
        )
        inputs = {key: value.to(model.device) for key, value in inputs.items()}
        with torch.no_grad():
            result = model(**inputs, labels=inputs["input_ids"])
        rows.append({
            "image_id": int(pair["image_id"]),
            "caption": pair["caption"],
            "label": int(pair["label"]),
            "pair_type": pair["pair_type"],
            "loss": float(result.loss.detach().cpu()),
        })

    elapsed = time.perf_counter() - start
    peak_memory_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
    del model, processor
    clear_gpu()
    return rows, elapsed, peak_memory_mb


def run_git(precision):
    """Évaluer la cohérence image-légende avec GIT."""
    from transformers import GitForCausalLM, GitProcessor

    model_id = "microsoft/git-base"
    processor = GitProcessor.from_pretrained(model_id)
    model = GitForCausalLM.from_pretrained(
        model_id,
        quantization_config=quantization_config(precision),
        torch_dtype=torch.float16 if precision == "fp16" else None,
        device_map="auto",
    )
    model.eval()
    torch.cuda.reset_peak_memory_stats()

    rows = []
    start = time.perf_counter()
    for _, pair in tqdm(pairs_df.iterrows(), total=len(pairs_df), desc=f"GIT {precision}"):
        example = sample[int(pair["image_id"])]
        inputs = processor(
            images=example["image"],
            text=pair["caption"],
            return_tensors="pt",
            padding=True,
        )
        inputs = {key: value.to(model.device) for key, value in inputs.items()}
        with torch.no_grad():
            result = model(**inputs, labels=inputs["input_ids"])
        rows.append({
            "image_id": int(pair["image_id"]),
            "caption": pair["caption"],
            "label": int(pair["label"]),
            "pair_type": pair["pair_type"],
            "loss": float(result.loss.detach().cpu()),
        })

    elapsed = time.perf_counter() - start
    peak_memory_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
    del model, processor
    clear_gpu()
    return rows, elapsed, peak_memory_mb

## Expérience 1/6 — BLIP en fp16 (précision normale)

In [4]:
def run_model(run_fn, model_name, precision):
    """Exécuter une expérience, mesurer le temps et la mémoire, sauvegarder."""
    path = output_path(model_name, precision)
    if path.exists():
        print(f"Déjà terminé, ignoré : {path.name}")
        return None

    clear_gpu()
    print(f"Démarrage : {model_name} - {precision}")
    judgements, elapsed, peak_memory_mb = run_fn(precision)
    judgements_df = pd.DataFrame(judgements)
    judgements_df["predicted_label"] = (
        judgements_df.groupby("image_id")["loss"]
        .rank(method="first", ascending=True)
        .eq(1)
        .astype(int)
    )
    accuracy = (judgements_df["predicted_label"] == judgements_df["label"]).mean()
    judgements_df.to_csv(path, index=False)

    row = {
        "model": model_name,
        "precision": precision,
        "accuracy": accuracy,
        "seconds": elapsed,
        "pairs": len(judgements_df),
        "seconds_per_pair": elapsed / len(judgements_df),
        "peak_memory_mb": peak_memory_mb,
    }
    summary_path = RESULTS_DIR / "model_judgement_summary.csv"
    if summary_path.exists():
        summary = pd.read_csv(summary_path)
        summary = summary[~((summary["model"] == model_name) & (summary["precision"] == precision))]
        summary = pd.concat([summary, pd.DataFrame([row])], ignore_index=True)
    else:
        summary = pd.DataFrame([row])
    summary.to_csv(summary_path, index=False)
    print(f"Sauvegardé : {path}")
    print(f"Accuracy : {accuracy:.3f} | Temps : {elapsed:.1f}s | Mémoire peak : {peak_memory_mb:.0f} Mo")
    clear_gpu()
    return row


row = run_model(run_blip, "blip", "fp16")
if row:
    display(pd.DataFrame([row]))

Déjà terminé, ignoré : judgements_blip_fp16.csv


## Expérience 2/6 — BLIP en int8 (quantification 8 bits)

In [5]:
row = run_model(run_blip, "blip", "int8")
if row:
    display(pd.DataFrame([row]))

Déjà terminé, ignoré : judgements_blip_int8.csv


## Expérience 3/6 — BLIP en int4 (quantification 4 bits)

In [6]:
row = run_model(run_blip, "blip", "int4")
if row:
    display(pd.DataFrame([row]))

Déjà terminé, ignoré : judgements_blip_int4.csv


## Expérience 4/6 — GIT en fp16 (précision normale)

In [7]:
row = run_model(run_git, "git", "fp16")
if row:
    display(pd.DataFrame([row]))

Déjà terminé, ignoré : judgements_git_fp16.csv


## Expérience 5/6 — GIT en int8 (quantification 8 bits)

In [8]:
row = run_model(run_git, "git", "int8")
if row:
    display(pd.DataFrame([row]))

Déjà terminé, ignoré : judgements_git_int8.csv


## Expérience 6/6 — GIT en int4 (quantification 4 bits)

In [9]:
row = run_model(run_git, "git", "int4")
if row:
    display(pd.DataFrame([row]))

Déjà terminé, ignoré : judgements_git_int4.csv


## Résumé de toutes les expériences

In [10]:
# Afficher le résumé disponible après chaque expérience.
summary_path = RESULTS_DIR / "model_judgement_summary.csv"
if summary_path.exists():
    display(pd.read_csv(summary_path))
else:
    print("Aucune expérience n'a encore été enregistrée.")

,model,precision,accuracy,seconds,pairs,seconds_per_pair,peak_memory_mb
0,blip,fp16,0.892,41.370285,1000,0.041370,464.233398
1,blip,int8,0.890,397.251435,1000,0.397251,356.850098
2,blip,int4,0.890,73.693434,1000,0.073693,266.889160
3,git,fp16,0.940,24.983541,1000,0.024984,386.181152
4,git,int8,0.940,284.691163,1000,0.284691,375.881348
5,git,int4,0.926,66.445382,1000,0.066445,320.871582


## Vérification

Les fichiers `judgements_blip_*.csv`, `judgements_git_*.csv` et `model_judgement_summary.csv` seront utilisés par le notebook 04. Les images et les modèles téléchargés restent dans Colab/Google Drive et ne doivent pas être ajoutés au dépôt GitHub.